# Learning from data: statistics and machine learning {#sec-learning}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c08-learning.ipynb)

So far the waiting-list model has been built from structure: what accumulates, what flows, which loops push back. Its numbers came from judgement and a glance at the data. The rest of the book asks harder questions of the data. How sure are we of a growth rate? How well does a model fit, and in what units? Could a forecasting model trained on the data do better than a feedback model, and when would it fail? This chapter gives you the statistics and machine learning those questions need, taught on the waiting list, and shows where each one connects to system dynamics.

**Who it is for.** Readers who have not studied statistics, or studied it long ago, and readers who know machine learning but not how it meets feedback models. **Who can skip what.** If you can explain a confidence interval, a least-squares fit and a train-test split, skim the table in section 7 and go on to chapter 9. **Prerequisites.** Stocks, flows and Euler's method (@sec-push-back), and the Python of @sec-python, especially NumPy and pandas. **Time.** About three hours with the exercises.

## Learning outcomes

By the end of this chapter you will be able to:

1. **Describe** a noisy series with its mean, standard deviation and percentiles, and **say** when the normal distribution is a fair summary (Try it 1).
2. **Attach** a confidence interval to an estimate, by formula and by bootstrap, and **explain** what it does and does not promise (Try it 2).
3. **Measure** association with a correlation and an autocorrelation, and **explain** why feedback makes correlation a poor guide to cause (Try it 3).
4. **Fit** a line and a growth rate by least squares and **read** the residuals (Try it 4).
5. **Choose** an error measure (RMSE, MAE, MAPE) and **report** it in units a decision maker understands (Try it 5).
6. **Train** and **test** a forecasting model without fooling yourself: split in time, avoid leakage, spot overfitting (Try it 6).
7. **Justify** when a feedback model, a statistical model or a learned model should answer a question, and how they combine (Try it 7).

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 12                          # one seed per chapter
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"]).set_index("month")

# Okabe-Ito colours (safe for colour-blind readers) with a distinct line style
# for each, so every figure also reads in greyscale
plt.rcParams["axes.prop_cycle"] = (
    cycler(color=["#000000", "#E69F00", "#56B4E9", "#009E73",
                  "#0072B2", "#D55E00", "#CC79A7"])
    + cycler(linestyle=["-", "--", ":", "-.", (0, (5, 1)),
                        (0, (3, 1, 1, 1)), (0, (1, 1))]))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3})
print("loopbricks", lb.__version__, "| data:", len(wl), "months")

## 1. Variation: describing a noisy series

Start with the net flow, the month-to-month change in the list (@sec-push-back). Before 2020 the system was roughly steady, so the six years 2014 to 2019 are a sample of what a "normal" month looks like.

In [ ]:
#| fig-alt: "Histogram of 71 monthly changes in the waiting list, 2014 to 2019, centred a little above zero, with a normal curve drawn over it."
net = wl["total"].diff().loc["2014-02":"2019-12"]     # 71 monthly changes
mean, sd = net.mean(), net.std()
print(f"months {len(net)}, mean {mean:.0f}, median {net.median():.0f}, "
      f"standard deviation {sd:.0f} patients a month")
print(f"5th, 50th and 95th percentiles: "
      f"{np.percentile(net, [5, 50, 95]).round(0)}")
inside = ((net > mean - sd) & (net < mean + sd)).mean()
inside2 = ((net > mean - 2 * sd) & (net < mean + 2 * sd)).mean()
print(f"share within 1 sd: {inside:.0%}; within 2 sd: {inside2:.0%}")

fig, ax = plt.subplots()
ax.hist(net, bins=15, density=True, color="#56B4E9", edgecolor="white",
        label="monthly change")
x = np.linspace(net.min() - 100, net.max() + 100, 200)
ax.plot(x, np.exp(-0.5 * ((x - mean) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
        label="normal curve, same mean and sd")
ax.set(xlabel="change in the list (patients a month)", ylabel="density")
ax.legend()
plt.show()

Figure: monthly changes in the waiting list, February 2014 to December 2019, with a normal distribution of the same mean and standard deviation drawn over them.

Three numbers summarise the series:

- The **mean** is the average, the sum divided by the count. The **median** is the middle value when the data are sorted; it is not pulled about by a few extreme months.
- The **standard deviation** (sd) measures spread: roughly, how far a typical month sits from the mean. It is the square root of the **variance**, the average squared distance from the mean, and it has the same units as the data.
- A **percentile** is the value below which that share of the data falls: the 5th percentile is beaten by 95% of months. The 5th and 95th together bracket the middle 90%.

The **normal distribution** is the bell curve. When many small, independent causes add up, as with thousands of separate referrals and appointments, the result tends to be normal, and then about 68% of values lie within one standard deviation of the mean and about 95% within two. The data here come close to that rule: 73% of months lie within one standard deviation of the mean and 96% within two. The rule fails for skewed quantities that cannot go below zero, such as waiting times, and for data with a few huge shocks, such as 2020. Always plot before you summarise.

**Try it 1.**

- *Check.* What is the standard deviation of the monthly change in 2020 and 2021? Why is a single standard deviation a poor summary there?
- *Apply.* The share of patients waiting over a year is a fraction between 0 and 1. Compute its mean and sd for 2014 to 2019 and say whether "mean plus or minus two sd" would give a sensible range.
- *Extend.* Draw 71 values from `rng.normal(mean, sd)` and compute their sd. Repeat 1,000 times. How much does the sd itself vary from sample to sample?

<details><summary>Worked answer</summary>

```python
shock = wl["total"].diff().loc["2020":"2021"]
print(round(shock.std()))          # far larger: one huge jump in 2020 dominates
share = (wl["w12_plus"] / wl["total"]).loc["2014":"2019"]
print(share.mean(), share.std())   # the range stays inside 0 to 1 here, but would not for a share near 0 or 1
sds = [rng.normal(mean, sd, 71).std(ddof=1) for _ in range(1000)]
print(np.percentile(sds, [5, 95]))  # the sd of a 71-month sample is itself uncertain by about +/- 15%
```

</details>

## 2. From a sample to a claim: confidence intervals

The mean change of 163 patients a month is an estimate from 71 months. Another 71 months of the same system would give a different number. A **confidence interval** says how far off the estimate might plausibly be. For a mean, the **standard error** is the standard deviation divided by the square root of the number of observations, and a 95% interval runs 1.96 standard errors either side. (1.96 is where the normal curve leaves 2.5% in each tail; for a 90% interval it is 1.645.)

In [ ]:
n = len(net)
se = sd / np.sqrt(n)
print(f"mean {mean:.0f}, standard error {se:.0f}")
print(f"95% interval {mean - 1.96 * se:.0f} to {mean + 1.96 * se:.0f}; "
      f"90% interval {mean - 1.645 * se:.0f} to {mean + 1.645 * se:.0f}")

# the bootstrap: resample the months with replacement, recompute, repeat
boot = np.array([rng.choice(net.to_numpy(), n).mean() for _ in range(5000)])
print(f"bootstrap 95% interval {np.percentile(boot, 2.5):.0f} to "
      f"{np.percentile(boot, 97.5):.0f}")

The formula gives a 95% interval of 112 to 213 patients a month, and the **bootstrap** agrees: 113 to 213. The bootstrap needs no formula. It treats the sample as a stand-in for the population, draws many new samples from it with replacement, and reads the interval off the spread of the recomputed estimates [@efron1979]. That makes it work for things with no textbook formula, such as a fitted model parameter, which is how chapter 10 uses it.

What a 95% interval promises: if you repeated the whole exercise many times, about 95% of the intervals you built would contain the true value. It does **not** promise that 95% of future months fall inside it; that is a much wider **prediction interval**, close to the mean plus or minus two standard deviations of the data, not of the mean.

Two cautions matter for feedback models. First, the formula assumes the months are independent. Section 3 shows they are not quite, and chapter 10 resamples blocks of months to respect that. Second, an interval covers only the noise in the data, not a wrong model. A precise estimate from a model with the wrong structure is precisely wrong.

One more number you will meet in chapter 10: **3.84**, which is 1.96 squared. When a model is fitted by minimising a sum of squared errors, the values of one parameter that raise the scaled error by less than 3.84 form its 95% interval. It is the same 1.96, applied to a squared quantity (statisticians call its distribution chi-square with one degree of freedom).

**Try it 2.**

- *Check.* How many months of data would halve the width of the 95% interval?
- *Apply.* Compute a bootstrap 90% interval for the **median** monthly change.
- *Extend.* Simulate the claim. Draw 71 values from `rng.normal(50, 150)` 2,000 times, build a 95% interval each time, and count how often it contains 50.

<details><summary>Worked answer</summary>

```python
print(4 * n)          # the width scales with 1 / sqrt(n): four times the data halves it
med = np.array([np.median(rng.choice(net.to_numpy(), n)) for _ in range(5000)])
print(np.percentile(med, [5, 95]))
hits = 0
for _ in range(2000):
    s = rng.normal(50, 150, 71)
    m, e = s.mean(), s.std(ddof=1) / np.sqrt(71)
    hits += (m - 1.96 * e < 50 < m + 1.96 * e)
print(hits / 2000)    # close to 0.95
```

</details>

## 3. Relationships: correlation, and why feedback fools it

The **correlation coefficient** $r$ measures how closely two series move together along a straight line, from $-1$ (perfectly opposite) through 0 (no straight-line relation) to $+1$ (perfectly together). An **autocorrelation** is the correlation of a series with itself a number of months earlier, its **lag**. It tells you whether this month's change carries information about next month's.

In [ ]:
def autocorr(x, lag):
    """Correlation of a series with itself `lag` steps earlier."""
    x = np.asarray(x, dtype=float)
    return np.corrcoef(x[:-lag], x[lag:])[0, 1]

for lag in (1, 2, 3, 12):
    print(f"lag {lag:>2}: autocorrelation of the monthly change "
          f"{autocorr(net, lag):+.2f}")
print(f"rough 95% band for 'no autocorrelation': +/- {1.96 / np.sqrt(n):.2f}")

The monthly changes are positively autocorrelated at lags 1 and 2 (+0.46 and +0.28), and again at lag 12 (+0.36), all above the band of ±0.23 that chance alone would give: a rising month tends to be followed by another, and a month tends to resemble the same month a year before, a hint of a yearly pattern. Bars inside about $\pm 1.96/\sqrt{n}$ are what pure chance would give; bars outside suggest structure.

Now the warning. In a system with feedback, correlation can point the wrong way or vanish when the cause is strong. Take the waiting-list model of @sec-push-back with a balancing loop: the longer the list, the more treatments, because the clinics book more patients when there is a queue. Feed it random referrals and ask the data what drives treatments.

In [ ]:
def waiting_list_run(months=240, capacity_per_patient=0.12, seed=SEED):
    """Monthly Euler run: referrals are random; treatments rise with the list."""
    r = np.random.default_rng(seed)
    referrals = 4000 + r.normal(0, 300, months)
    wl_, out = 33000.0, []
    for t in range(months):
        treatments = capacity_per_patient * wl_
        out.append((referrals[t], treatments, wl_))
        wl_ += referrals[t] - treatments
    return pd.DataFrame(out, columns=["referrals", "treatments", "list"])

sim = waiting_list_run()
print(sim.corr().round(2))

Referrals and treatments in the same month have a correlation of −0.06, as good as none, while treatments and the list correlate perfectly (1.00), because here one is a fixed share of the other. A naive reading would say referrals barely matter, yet every patient treated was first referred: in the long run treatments equal referrals exactly. The balancing loop absorbs each month's surprise into the stock, and the stock releases it slowly. The cause is real but delayed and smeared out through an accumulation, so a same-month correlation misses it. This is the main reason system dynamics insists on structure first: in a feedback system, the pattern of correlations is a product of the loops, not a map of them.

**Try it 3.**

- *Check.* Correlate referrals with treatments one, three and six months later in `sim`. Does a lag reveal the link?
- *Apply.* Add up referrals and treatments over the whole run. How do the totals compare, and what does that say about the near-zero correlation?
- *Extend.* Compute the lag-1 to lag-24 autocorrelation of the monthly change in `sim["list"]` and compare it with the data's.

<details><summary>Worked answer</summary>

```python
for lag in (1, 3, 6):
    print(lag, round(np.corrcoef(sim.referrals[:-lag], sim.treatments[lag:])[0, 1], 2))
print(round(sim.referrals.sum()), round(sim.treatments.sum()))
# 960,999 and 960,856: the totals differ by 143 patients, what is still on the list.
# Every referral is treated in the end; the loop only spreads the effect out in time.
ch = sim["list"].diff().dropna()
print([round(autocorr(ch, k), 2) for k in range(1, 25)][:6])
```

</details>

## 4. Fitting: least squares

**Fitting** means choosing a model's parameters so that its output is as close to the data as possible. **Least squares** measures closeness by the sum of squared **residuals**, the gaps between data and fit, and picks the parameters that make it smallest. Squaring treats overshoots and undershoots alike and punishes big misses most. For a straight line, `np.polyfit(x, y, 1)` does it exactly.

In [ ]:
#| fig-alt: "Waiting list 2014 to 2019 with a straight-line fit and an exponential fit on top; below, the residuals of the straight line wander above and below zero in long runs."
pre = wl["total"].loc[:"2019-12"]
t = np.arange(len(pre))                       # months since January 2014
slope, intercept = np.polyfit(t, pre.to_numpy(), 1)
line = intercept + slope * t
b, a = np.polyfit(t, np.log(pre.to_numpy()), 1)   # log-linear: exponential growth
expo = np.exp(a + b * t)
for name, f in [("straight line", line), ("exponential", expo)]:
    resid = pre.to_numpy() - f
    r2 = 1 - (resid ** 2).sum() / ((pre - pre.mean()) ** 2).sum()
    print(f"{name:>13}: R^2 = {r2:.3f}, residual sd {resid.std(ddof=2):.0f}")
print(f"line: {slope:.0f} patients a month; exponential: "
      f"{np.exp(12 * b) - 1:.1%} a year")

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(pre.index, pre / 1000, label="data")
axes[0].plot(pre.index, line / 1000, label="straight line")
axes[0].plot(pre.index, expo / 1000, label="exponential")
axes[0].set(ylabel="patients waiting (thousands)")
axes[0].legend()
axes[1].plot(pre.index, pre.to_numpy() - line)
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set(xlabel="month", ylabel="residual (patients)")
plt.tight_layout()
plt.show()

Figure: the waiting list from 2014 to 2019 with two least-squares fits (top), and the residuals of the straight line (bottom).

The straight line rises 158 patients a month and has an $R^2$ of 0.930; the exponential grows 5.2% a year, with an $R^2$ of 0.946. **R squared** ($R^2$) is the share of the variation in the data that the fit accounts for: 1 minus the sum of squared residuals over the sum of squared deviations from the mean. Both fits score well. That is the first lesson: a high $R^2$ on a trending series is easy and proves little.

The residuals tell more. They should look like noise: no trend, no pattern, no long runs on one side of zero. Here they wander in runs lasting a year or more. That says the data hold something a straight line does not, here the speeding-up in 2017 that the running case explains with growing referrals (@sec-loops).

The log-linear fit is a standard trick worth knowing. Exponential growth $y = e^{a + bt}$ becomes a straight line in $\log y$, so a straight-line fit to the logarithm gives the growth rate $b$ per month directly, and $e^{12b} - 1$ is the rate per year.

This is the bridge to calibration. A system dynamics model is also a function from parameters to a curve; it just has no formula, so the computer runs it for each candidate setting and searches for the least sum of squares. Chapter 10 does exactly that, with the same residual checks.

**Try it 4.**

- *Check.* Fit a straight line to 2022 and 2023 only. What slope do you get, and what does it say about the initiative?
- *Apply.* Fit the exponential to 2014 to 2016 and to 2017 to 2019 separately. Did growth speed up?
- *Extend.* Plot the residuals of the exponential fit and compare their pattern with the straight line's.

<details><summary>Worked answer</summary>

```python
late = wl["total"].loc["2022":"2023"]
print(np.polyfit(np.arange(len(late)), late.to_numpy(), 1)[0])   # about 43 a month, against 158 before 2020
for a_, b_ in [("2014", "2016"), ("2017", "2019")]:
    y = wl["total"].loc[a_:b_]
    print(a_, b_, np.exp(12 * np.polyfit(np.arange(len(y)), np.log(y.to_numpy()), 1)[0]) - 1)
fig, ax = plt.subplots(); ax.plot(pre.index, pre.to_numpy() - expo); ax.axhline(0); plt.show()
```

</details>

## 5. How wrong is a fit? Error measures

A single number for fit quality is useful for comparing models and for reporting. Each common choice answers a slightly different question.

| Measure | Formula | Units | Reads as |
|---|---|---|---|
| RMSE, root mean square error | $\sqrt{\text{mean}(e^2)}$ | the data's | a typical miss, with big misses weighted up |
| MAE, mean absolute error | $\text{mean}(\lvert e\rvert)$ | the data's | the average miss |
| MAPE, mean absolute percentage error | $\text{mean}(\lvert e/y\rvert)$ | percent | the average miss relative to the size of the data |

Here $e$ is the residual, data minus model, and $y$ the data.

In [ ]:
def errors(y, f):
    """RMSE, MAE and MAPE of a fit f to data y."""
    y, f = np.asarray(y, float), np.asarray(f, float)
    e = y - f
    return {"RMSE": np.sqrt((e ** 2).mean()), "MAE": np.abs(e).mean(),
            "MAPE": np.abs(e / y).mean()}

for name, f in [("straight line", line), ("exponential", expo)]:
    m = errors(pre, f)
    print(f"{name:>13}: RMSE {m['RMSE']:,.0f}, MAE {m['MAE']:,.0f} patients, "
          f"MAPE {m['MAPE']:.2%}")

# one bad month changes the picture
spoiled = pre.to_numpy().copy()
spoiled[40] += 5000
m = errors(spoiled, line)
print(f"one month 5,000 out: RMSE {m['RMSE']:,.0f}, MAE {m['MAE']:,.0f}")

The exponential is the better fit on every measure: RMSE 796 against 903 patients, MAE 643 against 737, MAPE 1.78% against 2.05%. RMSE is always at least as large as MAE, and the gap grows when a few misses are big: pushing one month 5,000 out raises the straight line's RMSE from 903 to 1,026 but its MAE only from 737 to 785. Use RMSE when big misses are what hurts, such as a forecast that runs out of beds, and MAE when every patient of error costs the same. MAPE is easy to explain ("on average 2% out") but misleads when the data come near zero, because dividing by a small number inflates it. For a decision maker, report the measure in the data's own units ("the model is typically 600 patients out on a list of 40,000") before any percentage.

**Try it 5.**

- *Check.* Compute RMSE, MAE and MAPE of a "no change" forecast that predicts each month's list as last month's, over 2014 to 2019.
- *Apply.* Why is that naive forecast so hard to beat a month ahead, and so easy to beat a year ahead? Test it with a 12-month version.
- *Extend.* Compute the MAPE of a model of the monthly *change* rather than the level. What goes wrong?

<details><summary>Worked answer</summary>

```python
y = pre.to_numpy()
print(errors(y[1:], y[:-1]))      # a month ahead, the list barely moves: small errors
print(errors(y[12:], y[:-12]))    # a year ahead, it has moved a lot: much larger errors
ch = np.diff(y)
print(errors(ch[1:], np.full(len(ch) - 1, ch.mean()))["MAPE"])
# changes pass close to zero, so dividing by them makes MAPE explode
```

</details>

## 6. Machine learning: learning a forecast from data

**Machine learning** is fitting taken further: a flexible model with many parameters learns a relationship from examples rather than from a formula someone wrote down. The vocabulary:

- **Target**: what you want to predict, here next month's list. **Features**: what the model may use to predict it, here the last few months.
- **Training**: fitting the parameters on part of the data. **Testing**: measuring the error on data the model has never seen. Only the test error says how the model will do on new cases.
- **Overfitting**: a model flexible enough to learn the noise as well as the signal fits the training data beautifully and the test data badly.
- **Leakage**: any route by which information from the test period reaches training, such as scaling with the mean of the whole series. Leakage makes a model look better than it is.

For time series the rule is simple: **split in time**. Train on the past, test on what came after, as a forecaster would have had to.

A clean way to see overfitting is to fit polynomials of rising degree to four years of the list and test them on the fifth.

In [ ]:
#| fig-alt: "Training error falls steadily as polynomial degree rises from 1 to 8; test error is lowest at a low degree and then rises steeply."
y = pre.to_numpy()
train, test = slice(0, 48), slice(48, 60)          # 2014 to 2017, then 2018
x = np.arange(len(y)) / 48                         # scaled time keeps the fit stable
rows = []
for deg in range(1, 9):
    coef = np.polyfit(x[train], y[train], deg)
    rows.append({"degree": deg,
                 "train RMSE": errors(y[train], np.polyval(coef, x[train]))["RMSE"],
                 "test RMSE": errors(y[test], np.polyval(coef, x[test]))["RMSE"]})
fit = pd.DataFrame(rows).set_index("degree")
print(fit.round(0).to_string())

fig, ax = plt.subplots()
ax.plot(fit.index, fit["train RMSE"], marker="o", label="training (2014 to 2017)")
ax.plot(fit.index, fit["test RMSE"], marker="s", label="test (2018)")
ax.set(xlabel="polynomial degree", ylabel="RMSE (patients)", yscale="log")
ax.legend()
plt.show()

Figure: error of polynomial trends of degree 1 to 8 fitted to 2014 to 2017 and tested on 2018. Note the logarithmic scale.

Training error falls from 639 patients at degree 1 to 269 at degree 8. Test error is lowest at degree 2, 203 patients, and then explodes, to 75,758 at degree 8: the high-degree curves swing wildly just beyond the data they were fitted to. Every extra degree buys a better fit to the past and, beyond a point, a worse forecast. Choosing the model by its test error, not its training error, is the core discipline of machine learning.

Now a model that learns from lagged values, the workhorse of data-driven forecasting. It predicts next month's change in the list from the last three changes, a linear **autoregression**, fitted by least squares with `np.linalg.lstsq`, which solves a least-squares problem with many features at once. More powerful learners (random forests, gradient boosting, neural networks) use the same setup with a more flexible function in the middle; scikit-learn provides them all with the same `fit` and `predict` pattern.

In [ ]:
def lagged(series, p=3):
    """Features: the last p monthly changes. Target: the next change."""
    d = np.diff(np.asarray(series, float))
    X = np.column_stack([d[p - k - 1:len(d) - k - 1] for k in range(p)])
    return np.column_stack([np.ones(len(X)), X]), d[p:]

X, target = lagged(y)
split = 47                                     # train to the end of 2017
coef, *_ = np.linalg.lstsq(X[:split], target[:split], rcond=None)
pred = X @ coef
naive = np.full(len(target), target[:split].mean())   # "same as the average"
print("one month ahead, 2018 to 2019:")
print(f"  autoregression RMSE {errors(target[split:], pred[split:])['RMSE']:.0f}")
print(f"  average change  RMSE {errors(target[split:], naive[split:])['RMSE']:.0f}")

On 2018 and 2019 the autoregression is typically 191 patients a month out, against 198 for simply predicting the average change of 2014 to 2017. That is a typical result on short, noisy series: a learned model must beat a simple benchmark on held-out data before anyone should use it, and here the margin is small. Chapter 12 runs a fuller contest between learned and feedback models.

**Try it 6.**

- *Check.* Refit the autoregression with 1, 6 and 12 lags. Which does best on the test years?
- *Apply.* Introduce leakage on purpose: centre the changes on the mean of all 2014 to 2019 before fitting. Does the test error flatter the model?
- *Extend.* With `p=12`, compare training and test RMSE. Is it overfitting?

<details><summary>Worked answer</summary>

```python
for p in (1, 6, 12):
    Xp, tp = lagged(y, p)
    s = 47 - (p - 3)                    # keep the test period the same months
    c, *_ = np.linalg.lstsq(Xp[:s], tp[:s], rcond=None)
    print(p, round(errors(tp[s:], Xp[s:] @ c)["RMSE"]))
Xp, tp = lagged(y, 12); s = 47 - 9
c, *_ = np.linalg.lstsq(Xp[:s], tp[:s], rcond=None)
print(errors(tp[:s], Xp[:s] @ c)["RMSE"], errors(tp[s:], Xp[s:] @ c)["RMSE"])
# training error well below test error: 13 parameters learn noise from 38 months
```

</details>

## 7. How statistics, machine learning and system dynamics connect

The three answer different questions, and the useful skill is knowing which question you are asking.

| Approach | Learns from | Answers best | Weak when |
|---|---|---|---|
| Statistics | data, with a stated model of the noise | how big, how sure, is the difference real | the structure behind the data changes |
| Machine learning | data, with a flexible function | what comes next if the world goes on as before | the future is unlike the training data, such as after a policy change |
| System dynamics | structure (stocks, flows, loops) checked against data | what happens if we change the system | parameters are poorly known or the boundary is wrong |

The weakness of learned models in the last row of the middle line is the one that matters most for policy. A model trained on history learns the correlations the old policy produced. Change the policy and those correlations change too, because the loops now run differently. The feedback model carries the structure, so it can be asked about a world that has never been observed. Here is the effect on simulated data where we know the truth: a list that grows for seven years, then gets 500 more treatments a month.

In [ ]:
#| fig-alt: "Simulated waiting list rising for 84 months, then falling after extra capacity; an autoregression trained on the first 84 months keeps forecasting a rise, while the stock-and-flow model with the new capacity follows the fall."
def true_system(months=120, policy_month=84, extra=500, seed=SEED):
    """Referrals grow 0.05% a month with noise; capacity jumps at policy_month."""
    r = np.random.default_rng(seed)
    t = np.arange(months)
    referrals = 4000 * 1.0005 ** t + r.normal(0, 150, months)
    capacity = 3950 + extra * (t >= policy_month)
    wl_, out = 32000.0, []
    for k in range(months):
        out.append(wl_)
        wl_ += referrals[k] - min(capacity[k], wl_ / 0.5)
    return np.array(out), referrals

truth, refs = true_system()
history = truth[:84]

# learned model: autoregression on the history, rolled forward 36 months
Xh, th = lagged(history, 3)
c, *_ = np.linalg.lstsq(Xh, th, rcond=None)
ml, d = list(history), list(np.diff(history)[-3:])
for _ in range(36):
    step = c[0] + c[1] * d[-1] + c[2] * d[-2] + c[3] * d[-3]
    d.append(step)
    ml.append(ml[-1] + step)

# feedback model: same structure, referral growth estimated from history, new capacity
g = np.polyfit(np.arange(84), np.log(refs[:84]), 1)
sd_model, wl_ = [], history[-1]
for k in range(84, 120):
    sd_model.append(wl_)
    wl_ += np.exp(g[1] + g[0] * k) - min(4450, wl_ / 0.5)

fig, ax = plt.subplots()
ax.plot(np.arange(120), truth / 1000, label="what happened (simulated truth)")
ax.plot(np.arange(84, 120), np.array(ml[84:]) / 1000,
        label="learned from history")
ax.plot(np.arange(84, 120), np.array(sd_model) / 1000,
        label="stock-and-flow model with the new capacity")
ax.axvline(84, color="grey", ls=":", lw=1)
ax.set(xlabel="month", ylabel="patients waiting (thousands)")
ax.legend()
plt.show()
print(f"month 119: truth {truth[-1]:,.0f}, learned {ml[-1]:,.0f}, "
      f"feedback model {sd_model[-1]:,.0f}")
print(f"RMSE over months 84 to 119: learned "
      f"{errors(truth[84:], ml[84:])['RMSE']:,.0f}, feedback model "
      f"{errors(truth[84:], sd_model)['RMSE']:,.0f}")

Figure: a simulated list (known truth) with extra capacity from month 84 (dotted line). The learned model extends the old trend; the stock-and-flow model, given the new capacity, follows the fall.

At month 119 the simulated list is 35,651. The learned model, which saw only the rising years, says 49,266; the stock-and-flow model says 36,046. Over the three years after the change the learned model is typically 8,438 patients out, the feedback model 536. The learned model is not badly built; it is answering a different question, what happens if nothing changes, and the question here was what happens if something does.

The approaches are partners, not rivals, and the rest of the book uses them together:

- **Statistics estimates the parameters** of a feedback model and states how sure we are: calibration, residual checks and bootstrap bands in chapter 10.
- **Statistics tests the model**: behaviour tests compare model and data with the measures of section 5 (chapter 9).
- **Machine learning forecasts what the model takes as given**, such as referrals driven by demography, and the feedback model turns those inputs into consequences.
- **Machine learning helps find structure and speed up models**: equation discovery, hybrid models with learned pieces and fast surrogates, all in chapter 12.
- **System dynamics tells machine learning what cannot be learned** from history alone: the response to a change nobody has tried.

**Try it 7.**

- *Check.* Rerun `true_system` with `extra=0`, so nothing changes. Over months 84 to 119, which forecast is closer now?
- *Apply.* Give the feedback model a wrong capacity, 4,350 instead of 4,450. How far off is it at month 119? Which is the bigger risk, the wrong structure or a wrong parameter?
- *Extend.* Add the capacity as a feature to the autoregression (a column that is 3,950 or 4,450). Can it learn the effect from the history? Why not?

<details><summary>Worked answer</summary>

With `extra=0` the learned model's RMSE drops from 8,438 to 1,784: it does far better when the future is like the past. The feedback model still wins here, but only because it was given the true structure, which a real model never has exactly. With a capacity 100 a month too low, the feedback model ends 3,895 out at month 119, against 13,616 for the learned model: a wrong parameter costs less than the wrong structure, because the direction is still right. The capacity column is constant at 3,950 throughout the history, so it carries no information: a learned model cannot estimate the effect of something that never varied. Only structure, or an experiment, can say what extra capacity does.

</details>

## Check it

Habits to carry into every chapter that uses data:

- **Plot before you summarise.** A mean and a standard deviation hide shocks, trends and skew.
- **Say what an interval covers.** A confidence interval for a parameter is not a range for next month.
- **Read residuals, not just $R^2$.** Patterned residuals mean the model is missing something.
- **Test out of sample, split in time.** Anything else lets the future leak into the past.
- **Beat a naive benchmark** ("same as last month", "same as the average") before trusting a learned model.

## Running case

The running case gains two numbers with honest uncertainty. From 2014 to 2019 the list rose by 163 patients (95% interval 112 to 213) a month on average, and the exponential fit puts its growth at 5.2% a year. Neither is the referral growth rate itself: the list is a stock, and it grows by the *gap* between referrals and treatments, which is why chapter 10 calibrates a model with both flows rather than fitting a curve to the stock. Chapter 9 first tests whether the model's structure deserves that effort.

## Tell the decision maker

"The list grew by about 160 patients a month before the pandemic, give or take 50. A forecast that simply extends the trend is a fair guide only if nothing changes; to judge a policy, such as extra clinics, we need the model of how patients flow, because past data cannot show the effect of something that has not been tried."

## Going further

- **Regression with many predictors**, standard errors of coefficients and hypothesis tests are the next step in statistics; @james2021 is a free and readable introduction that also covers the main machine learning methods.
- **Forecasting** with seasonal models, exponential smoothing and proper time-series cross-validation is covered in @hyndman2021, also free online.
- **scikit-learn** gives random forests, gradient boosting and neural networks the same `fit` and `predict` interface; swap one in for the autoregression of section 6 and compare test errors.
- **Bayesian calibration** treats parameters as distributions and combines data with prior knowledge; chapter 10 points to it.

## Mini-project

Using the 2014 to 2019 data only, build three one-year-ahead forecasts of the list for 2019: the "same as last year" benchmark, the exponential trend of section 4, and an autoregression with lags of your choice trained on 2014 to 2017 and validated on 2018. Report each forecast's RMSE and MAE for 2019 in patients, with a bootstrap 90% interval for the RMSE, and write three sentences for a manager on which to use and when it would stop working.

## Further reading

- @james2021, on statistical learning, from regression to modern machine learning.
- @hyndman2021, on forecasting principles and practice for time series.
- @efron1979, the original bootstrap paper.
- @sterman2000, chapter 21, on statistics for comparing models with data.